# Run the WINNER's code (cuda-torso) on small-graph — Kaggle GPU

This compiles and runs the leaderboard-winning solution **directly** (their
`run.py` + `libeval.cu`), from random, at scale. It *is* the SOTA method — no
reimplementation. On small (n=1357) a long session can approach ~100k generations,
which is what produced their leaderboard score.

**Settings → GPU (P100 or T4), Internet ON. Add Data → upload `cuda-torso-main.zip`**
(the winner's repo you already have). Then Run All.

In [ ]:
import os, glob, zipfile, shutil
shutil.rmtree('/kaggle/working/cuda', ignore_errors=True); os.makedirs('/kaggle/working/cuda', exist_ok=True)
z = glob.glob('/kaggle/input/**/cuda-torso*.zip', recursive=True)
ex = glob.glob('/kaggle/input/**/libeval.cu', recursive=True)
if z: zipfile.ZipFile(z[0]).extractall('/kaggle/working/cuda')
elif ex: shutil.copytree(ex[0].rsplit('/libeval.cu',1)[0], '/kaggle/working/cuda/cuda-torso-main')
else: raise SystemExit("Upload cuda-torso-main.zip via Add Data.")
ROOT = os.path.dirname(glob.glob('/kaggle/working/cuda/**/libeval.cu', recursive=True)[0])
os.chdir(ROOT); print('cwd:', ROOT, '| files:', os.listdir('.'))

In [ ]:
# compile their CUDA evaluator -> libeval.so
!nvcc -shared -Xcompiler -fPIC -o libeval.so libeval.cu && echo "compiled libeval.so" && ls -la libeval.so
import torch; print('torch CUDA:', torch.cuda.is_available())

## Run their search on small-graph
Their defaults: batch 1024, up to 100k generations, checkpoints every 50 gens to
`submissions/small-graph/<hvi>.json` (filename = the score; more negative = better).
It runs until the session limit. The best file at the end is your result.

In [ ]:
!python3 run.py --graph small-graph --batch_size 1024 --max_generations 100000 --checkpoint_every 50 --log_every 25

## Collect the best
Each checkpoint writes `submissions/small-graph/<hvi>.json` (hvi = score). Pick the
most-negative filename = best front found. Copy it out and download it.

In [ ]:
import glob, os, shutil
subs = glob.glob('submissions/small-graph/*.json')
# filename is the (negative) hvi; most-negative = best
best = min(subs, key=lambda f: int(os.path.basename(f).split('.')[0])) if subs else None
print('checkpoints:', len(subs), '| best:', best)
if best: shutil.copy(best, '/kaggle/working/cuda_small_best.json'); print('saved /kaggle/working/cuda_small_best.json — leaderboard target small = -1,829,919')